# Filter AMR papers with LLM

Use the `llm.py` helper to ask a model whether a paper's abstract asserts that a pathogen is showing **significant resistance** to a given antibiotic.


In [1]:

# --- LLM filtering configuration ---
import pandas as pd

from llm import create_llm_client
from amr.filtering import (
    DEFAULT_SYSTEM_PROMPT,
    prepare_pmids_to_review,
    run_llm_filter,
)

SYSTEM_PROMPT = """
You are an expert microbiologist reviewing evidence. Given a title and abstract, decide whether the authors explicitly state or conclude that the specified pathogen is showing clinically meaningful resistance to the specified antibiotic.

Rules:
- Answer "yes" only if the abstract clearly reports resistance, reduced susceptibility, treatment failure due to resistance, or emergence of resistant isolates for that pathogen/antibiotic pair.
- Answer "no" if the abstract does not make that claim, is about a different organism/drug, only mentions testing without resistance, or only provides background.
- Answer "uncertain" if the abstract is missing, ambiguous, or only indirectly related.
- Base the decision only on the provided text; do not infer beyond it.

Return compact JSON with keys: decision (yes/no/uncertain) and rationale (one short sentence).
"""

LLM_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
NCBI_EMAIL = "your_email@example.com"
N_PMIDS_PER_RUN = 1000  # how many unprocessed PMIDs to handle this run
RATE_LIMIT = 0.35  # seconds between PubMed calls
OUTPUT_PATH = "pmids_results_filtered.parquet"
LOG_PATH = "pmid_filter.log"
MAX_LLM_CALLS = None  # set to an integer to cap spend per run
DRY_RUN = False  # set True to skip LLM calls
SAVE_EVERY = 50  # checkpoint frequency

llm_kwargs = {}
if MODEL_NAME:
    llm_kwargs["model"] = MODEL_NAME

llm = create_llm_client(provider=LLM_PROVIDER, **llm_kwargs)
print(f"Using provider={LLM_PROVIDER}, model={MODEL_NAME or 'provider default'}; dry_run={DRY_RUN}; max_llm_calls={MAX_LLM_CALLS}")


Using provider=openai, model=gpt-4o-mini; dry_run=False; max_llm_calls=None


In [2]:
#!pip install openai requests pandas lxml

In [3]:

import os
import pandas as pd
from amr.filtering import prepare_pmids_to_review

# Load previous decisions
if os.path.exists(OUTPUT_PATH):
    reviewed_df = pd.read_parquet(OUTPUT_PATH)
    reviewed = set(reviewed_df['pmid'].astype(str)) if not reviewed_df.empty else set()
    print(f"Resuming with {len(reviewed)} PMIDs already reviewed from {OUTPUT_PATH}")
else:
    reviewed_df = None
    reviewed = set()

pmid_df = pd.read_parquet("pmids_results.parquet")
# Build full exploded table to report totals
full_long_df = prepare_pmids_to_review(pmid_df, exclude_pmids=None)
full_total = len(full_long_df)

# Exclude reviewed first, THEN cap to the next chunk
long_df = full_long_df
if reviewed:
    long_df = long_df[~long_df['pmid'].astype(str).isin(reviewed)]

pmid_total = len(long_df)
reviewed_count = len(reviewed)
print(f"Total PMIDs in source (exploded): {full_total}")
print(f"Loaded {pmid_total} candidate PMIDs after exclusion; reviewed set size: {reviewed_count}")

if N_PMIDS_PER_RUN:
    long_df = long_df.head(N_PMIDS_PER_RUN)

if len(long_df) == 0:
    print("No PMIDs remain after excluding reviewed. Consider using a fresh OUTPUT_PATH or adding more source data.")

print(f"PMIDs remaining to review this run: {len(long_df)}")
long_df.head()


Resuming with 8448 PMIDs already reviewed from pmids_results_filtered.parquet
Total PMIDs in source (exploded): 518057
Loaded 430266 candidate PMIDs after exclusion; reviewed set size: 8448
PMIDs remaining to review this run: 1000


,pmid,pathogen,antibiotic,start_date,end_date
15271,25932248,Acinetobacter baumannii,Gentamicin,2015/01/01,2019/12/31
15272,25860039,Acinetobacter baumannii,Gentamicin,2015/01/01,2019/12/31
15274,25599131,Acinetobacter baumannii,Gentamicin,2015/01/01,2019/12/31
15276,25543268,Acinetobacter baumannii,Gentamicin,2015/01/01,2019/12/31
15277,24827985,Acinetobacter baumannii,Gentamicin,2015/01/01,2019/12/31


# How decisions are derived

Each PMID is fetched and passed to the LLM with a precise prompt:

- Target pathogen and antibiotic are specified as “exact match only”.
- The LLM is instructed to decide if the authors explicitly state clinically meaningful resistance of THIS pathogen to THIS antibiotic.
- It must choose one of three decisions and return JSON.

# Decision meanings

**yes:** The abstract clearly reports resistance or reduced susceptibility for the specified pathogen/antibiotic pair—e.g., treatment failure due to resistance, emergence of resistant isolates, MICs showing resistance, or wording that unambiguously claims resistance for that pair.

**no:** The abstract does not make that claim for the specified pair. This includes:

- The study is about a different organism or drug.
- It mentions testing without reporting resistance for the target pair.
- It only provides background or context without asserting resistance for the target pair.

**uncertain:** Used when the evidence is insufficient or unusable:

- Missing or empty abstract.
- Ambiguous wording that doesn’t clearly assert or deny resistance for the target pair.
- Indirectly related content (e.g., adjacent species/drugs, surveillance without explicit resistance claim).
- LLM parsing/response failures or dry-run mode.

# Behavior in the pipeline

- PMIDs with no abstract are recorded as uncertain with rationale “no abstract”.
- LLM call failures are recorded as uncertain with a failure rationale.
- Dry runs mark entries as uncertain with rationale “dry_run”.
- Only PMIDs already present in the reviewed parquet are skipped entirely; all others get a decision recorded.

In [4]:
%%time
filtered_df = run_llm_filter(
    long_df,
    llm,
    email=NCBI_EMAIL,
    rate_limit=RATE_LIMIT,
    save_every=SAVE_EVERY,
    system_prompt=SYSTEM_PROMPT,
    progress_every=10,
    resume_from=reviewed_df,
    seen_pmids=reviewed,
    output_path=OUTPUT_PATH,
    log_path=LOG_PATH,
    max_llm_calls=MAX_LLM_CALLS,
    dry_run=DRY_RUN,
)
filtered_df.head()


[2025-11-23T13:00:07] [15272/1000] PMID 25932248 | Acinetobacter baumannii vs Gentamicin | interval 2015/01/01–2019/12/31
[2025-11-23T13:00:08]     decision=yes | rationale=The abstract reports gentamicin-resistant Acinetobacter baumannii isolates and the presence of resistance-related enzyme
[2025-11-23T13:00:09] [15273/1000] PMID 25860039 | Acinetobacter baumannii vs Gentamicin | interval 2015/01/01–2019/12/31
[2025-11-23T13:00:10]     decision=no | rationale=The abstract does not explicitly state resistance of Acinetobacter baumannii to gentamicin.
[2025-11-23T13:00:11] [15275/1000] PMID 25599131 | Acinetobacter baumannii vs Gentamicin | interval 2015/01/01–2019/12/31
[2025-11-23T13:00:12]     decision=yes | rationale=The abstract states that all but one strain of Acinetobacter baumannii were resistant to gentamicin.
[2025-11-23T13:00:13] [15277/1000] PMID 25543268 | Acinetobacter baumannii vs Gentamicin | interval 2015/01/01–2019/12/31
[2025-11-23T13:00:14]     decision=yes | ratio

,pmid,decision,rationale,pathogen,antibiotic,title,abstract,start_date,end_date
0,27544979,no,The abstract discusses Achromobacter xylosoxid...,Achromobacter denitrificans,Spectinomycin,Properties of Achromobacter xylosoxidans highl...,We herein discovered a highly resistant clinic...,2015/01/01,2019/12/31
1,15004037,no,The abstract mentions resistance to spectinomy...,Acinetobacter baumannii,Spectinomycin,Molecular epidemiology of sequential outbreaks...,The molecular epidemiology of multidrug-resist...,2000/01/01,2004/12/31
2,19361944,no,The abstract does not explicitly state resista...,Acinetobacter baumannii,Spectinomycin,Genetic basis of resistance to aminoglycosides...,A total of 75 Acinetobacter isolates resistant...,2005/01/01,2009/12/31
3,19028622,no,The abstract does not mention Acinetobacter ba...,Acinetobacter baumannii,Spectinomycin,Nosocomial spread of armA-mediated high-level ...,"1,310 Enterobacteriaceae, 242 Pseudomonas aeru...",2005/01/01,2009/12/31
4,16000417,yes,"The abstract mentions gene aadA1, which confer...",Acinetobacter baumannii,Spectinomycin,Detection and typing of integrons in epidemic ...,Integrons were sought in Acinetobacter isolate...,2005/01/01,2009/12/31


In [5]:

print("Summary (this run):")

reviewed_pmids = set()
if 'reviewed_df' in globals() and reviewed_df is not None and not reviewed_df.empty:
    reviewed_pmids = set(reviewed_df['pmid'].astype(str))

new_df = filtered_df.copy()
if reviewed_pmids:
    new_df = new_df[~new_df['pmid'].astype(str).isin(reviewed_pmids)]

run_total = len(new_df)
run_yes = (new_df['decision'].str.lower() == 'yes').sum()
run_no = (new_df['decision'].str.lower() == 'no').sum()
run_unc = (new_df['decision'].str.lower() == 'uncertain').sum()
run_fraction = run_yes / run_total if run_total else 0
print(f"Processed this run: {run_total} abstracts; yes {run_yes}, no {run_no}, uncertain {run_unc} (yes fraction {run_fraction:.3f}).")

all_total = len(filtered_df)
all_yes = (filtered_df['decision'].str.lower() == 'yes').sum()
all_no = (filtered_df['decision'].str.lower() == 'no').sum()
all_unc = (filtered_df['decision'].str.lower() == 'uncertain').sum()
all_fraction = all_yes / all_total if all_total else 0
print(f"Cumulative: {all_total} abstracts; yes {all_yes}, no {all_no}, uncertain {all_unc} (yes fraction {all_fraction:.3f}).")

if os.path.exists(OUTPUT_PATH):
    tail = open(LOG_PATH, 'r', encoding='utf-8').read().splitlines()
    tail = tail[-10:] if tail else []
    print("Log tail:")
    for line in tail:
        print(line)


Summary (this run):
Processed this run: 778 abstracts; yes 159, no 616, uncertain 3 (yes fraction 0.204).
Cumulative: 9226 abstracts; yes 2254, no 6772, uncertain 200 (yes fraction 0.244).
Log tail:
[2025-11-23T13:24:18] [16558/1000] PMID 33859725 | Clostridioides difficile vs Gentamicin | interval 2020/01/01–2024/12/31
[2025-11-23T13:24:19]     decision=no | rationale=The abstract does not mention resistance of Clostridioides difficile to gentamicin.
[2025-11-23T13:24:19] [16559/1000] PMID 33704685 | Clostridioides difficile vs Gentamicin | interval 2020/01/01–2024/12/31
[2025-11-23T13:24:21]     decision=no | rationale=The abstract does not mention resistance of Clostridioides difficile to gentamicin.
[2025-11-23T13:24:22] [16560/1000] PMID 6998374 | Clostridium botulinum vs Gentamicin | interval 1980/01/01–1984/12/31
[2025-11-23T13:24:23]     decision=yes | rationale=The abstract states that C. botulinum was not susceptible to gentamicin.
[2025-11-23T13:24:23] [16562/1000] PMID 3568

In [6]:

print(f"Saved results to {OUTPUT_PATH}")
filtered_df['decision'].value_counts()


Saved results to pmids_results_filtered.parquet


decision
no           6772
yes          2254
uncertain     200
Name: count, dtype: int64

In [7]:

# Build positive-only dataset and save
pmid_df_full = pd.read_parquet("pmids_results.parquet")

# Merge decisions onto exploded PMIDs
positive_flags = filtered_df[filtered_df["decision"].str.lower() == "yes"][["pmid"]].copy()
positive_rows = (
    pmid_df_full.explode("pmids")
    .assign(pmid=lambda d: d["pmids"].astype(str).str.extract(r"(\d+)")[0])
    .dropna(subset=["pmid"])
    .merge(positive_flags, on="pmid", how="inner")
)

# Reassemble to original grain
pmids_positive = (
    positive_rows.groupby(["pathogen", "antibiotic", "start_date", "end_date"], as_index=False)
    .agg(
        pmids=("pmid", list),
        count=("pmid", "size"),
        venue_histogram=("venue_histogram", "first"),
    )
)

positive_path = "pmids_results_positive.parquet"
pmids_positive.to_parquet(positive_path, index=False)
print(f"Saved positive-only dataset to {positive_path} with {len(pmids_positive)} rows")
pmids_positive.head()


Saved positive-only dataset to pmids_results_positive.parquet with 6519 rows


,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram
0,Achromobacter denitrificans,Amikacin,1985/01/01,1989/12/31,[3163242],1,"{""Antimicrobial agents and chemotherapy"": 1}"
1,Achromobacter denitrificans,Amikacin,2005/01/01,2009/12/31,[19795625],1,"{""Mikrobiyoloji bulteni"": 1}"
2,Achromobacter denitrificans,Amikacin,2015/01/01,2019/12/31,[26935636],1,"{""Medicine"": 1, ""The Japanese journal of antib..."
3,Achromobacter denitrificans,Amoxicillin,2005/01/01,2009/12/31,"[19795625, 17852927]",2,"{""Mikrobiyoloji bulteni"": 1, ""Scandinavian jou..."
4,Achromobacter denitrificans,Amoxicillin/clavulanic-acid,2005/01/01,2009/12/31,[17852927],1,"{""Scandinavian journal of infectious diseases""..."


In [8]:
pmids_positive[pmids_positive['count']>50]

,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram
98,Acinetobacter baumannii,Amikacin,2015/01/01,2019/12/31,"[32313533, 32032023, 31911917, 31706742, 31533...",82,"{""Annals of burns and fire disasters"": 2, ""Jou..."
99,Acinetobacter baumannii,Amikacin,2020/01/01,2024/12/31,"[39958989, 39766586, 39666200, 39418529, 39081...",78,"{""Frontiers in cellular and infection microbio..."
228,Acinetobacter baumannii,Gentamicin,2015/01/01,2019/12/31,"[32032023, 31911917, 31707391, 31667355, 31663...",62,"{""Journal of infection in developing countries..."
229,Acinetobacter baumannii,Gentamicin,2020/01/01,2024/12/31,"[39958989, 39766586, 39666200, 39464903, 39160...",60,"{""Frontiers in cellular and infection microbio..."
2205,Escherichia coli,Amikacin,2010/01/01,2014/12/31,"[25424607, 25379369, 25310915, 25285496, 25246...",67,"{""Journal of Nepal Health Research Council"": 2..."
2206,Escherichia coli,Amikacin,2015/01/01,2019/12/31,"[32045373, 31894059, 31893041, 31855054, 31683...",71,"{""Open access Macedonian journal of medical sc..."
2207,Escherichia coli,Amikacin,2020/01/01,2024/12/31,"[40673002, 39676194, 39619726, 39531253, 39518...",83,"{""JAC-antimicrobial resistance"": 8, ""Public he..."
2803,Klebsiella pneumoniae,Amikacin,2015/01/01,2019/12/31,"[32313533, 32058985, 31921701, 31855054, 31609...",52,"{""Annals of burns and fire disasters"": 1, ""Jou..."
2804,Klebsiella pneumoniae,Amikacin,2020/01/01,2024/12/31,"[39766529, 39734766, 39716087, 39678907, 39547...",90,"{""Archives of Razi Institute"": 5, ""Heliyon"": 3..."
4323,Pseudomonas aeruginosa,Amikacin,2010/01/01,2014/12/31,"[25429232, 25163250, 25141317, 24928215, 24899...",56,"{""Journal of preventive medicine and hygiene"":..."
